# Análise Exploratória — Gapminder

Análise de dados socioeconômicos de 142 países entre 1952 e 2007.

**Técnicas:** `groupby().agg()`, `transform()`, `apply()`, `rolling()`, NumPy vetorizado, matplotlib, seaborn, plotly

**Dataset:** [Gapminder via Jenny Bryan](https://github.com/jennybc/gapminder)

## 0. Imports e carregamento dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

url = 'https://raw.githubusercontent.com/jennybc/gapminder/master/inst/extdata/gapminder.tsv'
df = pd.read_csv(url, sep='\t')

print(f'Shape: {df.shape}')
print(f'Países: {df["country"].nunique()}')
print(f'Anos: {sorted(df["year"].unique())}')
df.head()

## 1. GDP per capita por continente

**Técnica:** `groupby().agg()` com múltiplas métricas em uma única operação.

Calculamos média, máximo e mínimo histórico do GDP per capita para cada continente.

In [ ]:
agg_gdpPercap = df.groupby('continent').agg(
    media_gdpPercap=('gdpPercap', 'mean'),
    maximo_gdpPercap=('gdpPercap', 'max'),
    minimo_gdpPercap=('gdpPercap', 'min'),
)
agg_gdpPercap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bars = agg_gdpPercap['media_gdpPercap'].sort_values().plot(
    kind='bar', ax=ax, color='steelblue'
)

# Valores em cima de cada barra
for p in ax.patches:
    ax.annotate(
        f'${p.get_height():,.0f}',
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

ax.set_title('Média do GDP per Capita por Continente (1952–2007)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Continente', fontsize=12)
ax.set_ylabel('GDP per Capita Médio (USD)', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('\nInsight: Oceania e Europa têm GDP médio histórico ~8x e ~6x maior que a África.')

## 2. Normalização do GDP por continente

**Técnica:** `groupby().transform()` — normaliza cada valor dentro do seu continente,
preservando o shape original do DataFrame.

Resultado: z-score por continente (média=0, desvio=1).

In [ ]:
df['gdpPercap_normalizado'] = df.groupby('continent')['gdpPercap'].transform(
    lambda x: (x - x.mean()) / x.std()
)

print('Primeiras linhas com GDP normalizado:')
df[['country', 'continent', 'year', 'gdpPercap', 'gdpPercap_normalizado']].head(10)

## 3. Variação percentual da expectativa de vida por país

**Técnica:** `groupby().apply()` com função customizada.

Calculamos quanto a expectativa de vida cresceu (ou diminuiu) em cada país
entre o primeiro e o último ano disponível no dataset.

In [ ]:
variacao_pct = df.groupby('country')['lifeExp'].apply(
    lambda x: ((x.iloc[-1] - x.iloc[0]) / x.iloc[0] * 100)
).rename('variacao_pct_lifeExp')

print('Top 10 maiores crescimentos:')
print(variacao_pct.sort_values(ascending=False).head(10))
print('\nPaíses com queda na expectativa de vida:')
print(variacao_pct[variacao_pct < 0].sort_values())

## 4. Média móvel da expectativa de vida — Brasil

**Técnica:** `rolling(window=3)` — média dos 3 períodos anteriores.

Os dois primeiros valores são `NaN` porque não há períodos suficientes para calcular
a janela de 3 pontos — comportamento esperado e correto.

In [ ]:
df_brasil = df[df['country'] == 'Brazil'].copy()
df_brasil['media_movel_3'] = df_brasil['lifeExp'].rolling(window=3).mean()

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df_brasil['year'], df_brasil['lifeExp'],
        marker='o', label='Expectativa de vida', color='steelblue', linewidth=2)
ax.plot(df_brasil['year'], df_brasil['media_movel_3'],
        linestyle='--', marker='o', label='Média móvel 3 períodos',
        color='tomato', linewidth=2)

# Anotação explicando o início da média móvel
ax.annotate(
    'Média móvel inicia\nno 3º período (1962)',
    xy=(1962, 53.3), xytext=(1970, 51.5),
    arrowprops=dict(arrowstyle='->', color='gray'),
    fontsize=9, color='gray'
)

ax.set_title('Expectativa de Vida no Brasil (1952–2007)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Anos', fontsize=12)
ax.legend(frameon=False, fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(df_brasil[['year', 'lifeExp', 'media_movel_3']].to_string(index=False))

## 5. Correlação entre variáveis

**Técnica:** `seaborn.heatmap()` com valores anotados.

- GDP per capita e expectativa de vida têm correlação moderada (r = 0.58)
- População praticamente não correlaciona com expectativa de vida (r = 0.06)

In [ ]:
corr = df[['lifeExp', 'pop', 'gdpPercap']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='coolwarm',
    square=True,
    linewidths=0.5,
    ax=ax,
    xticklabels=['Exp. de Vida', 'População', 'GDP per Capita'],
    yticklabels=['Exp. de Vida', 'População', 'GDP per Capita']
)

ax.set_title('Correlação entre Expectativa de Vida, População e GDP per Capita',
             fontsize=12, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

## 6. Scatter: GDP × Expectativa de Vida em 2007

**Técnica:** `matplotlib.scatter` com cores por continente.

Escala logarítmica no eixo X para melhor distribuição visual dos pontos
(GDP per capita tem distribuição fortemente assimétrica à direita).

In [ ]:
df_2007 = df[df['year'] == 2007]
continentes = df_2007['continent'].unique()
cores = ['steelblue', 'tomato', 'seagreen', 'mediumpurple', 'darkorange']

fig, ax = plt.subplots(figsize=(10, 6))

for continente, cor in zip(continentes, cores):
    dados = df_2007[df_2007['continent'] == continente]
    ax.scatter(dados['gdpPercap'], dados['lifeExp'],
               label=continente, color=cor, alpha=0.7, s=60)

ax.set_xscale('log')  # escala log para melhor distribuição
ax.set_title('GDP per Capita vs Expectativa de Vida em 2007 por Continente',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('GDP per Capita (escala logarítmica)', fontsize=12)
ax.set_ylabel('Expectativa de Vida (anos)', fontsize=12)
ax.legend(title='Continente', frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

## 7. Evolução da expectativa de vida por continente

**Técnica:** `groupby()` + matplotlib multi-linha.

A estagnação da África nos anos 90 é visível — possivelmente associada
ao impacto da epidemia de HIV/AIDS no continente.

In [ ]:
df_continente = df.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))

for continente in df_continente['continent'].unique():
    dados = df_continente[df_continente['continent'] == continente]
    ax.plot(dados['year'], dados['lifeExp'],
            marker='o', label=continente, linewidth=2)

ax.set_title('Evolução da Expectativa de Vida Média por Continente (1952–2007)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Expectativa de Vida Média (anos)', fontsize=12)
ax.legend(title='Continente', bbox_to_anchor=(1.02, 1), frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

## 8. Scatter interativo — Plotly

**Técnica:** `plotly.express.scatter()` com hover interativo.

Passe o mouse sobre os pontos para ver o país, GDP e expectativa de vida.
Tamanho dos pontos proporcional à população.

In [ ]:
fig = px.scatter(
    df_2007,
    x='gdpPercap',
    y='lifeExp',
    color='continent',
    size='pop',
    hover_name='country',
    hover_data={'gdpPercap': ':,.0f', 'lifeExp': ':.1f', 'pop': ':,'},
    log_x=True,
    title='PIB per Capita vs Expectativa de Vida em 2007 (interativo)',
    labels={
        'gdpPercap': 'PIB per Capita (escala log)',
        'lifeExp': 'Expectativa de Vida (anos)',
        'continent': 'Continente',
        'pop': 'População'
    }
)

fig.update_layout(
    plot_bgcolor='white',
    font_family='Arial',
    hoverlabel=dict(bgcolor='white')
)
fig.update_xaxes(showgrid=True, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridcolor='lightgray')

fig.show()